# Reference analysis notebook

This notebook is provided as reference analysis code for the accepted paper figures. It is not intended to be a standalone reproduction package. Model weights, LoRA adapters, datasets, and intermediate hidden-state files are not included. Local paths under `data/`, `adapters/`, and `outputs/` should be adjusted to the user's environment.

For the baseline-vs-ours heatmap, rows correspond to baseline layers and columns correspond to ours layers.


In [ ]:
# Figures: SVCCA similarity heatmaps for baseline, ours, and baseline-vs-ours layers.
# Required inputs: either saved hidden states or model specs that can extract hidden states.
# The notebook computes SVCCA scores for selected layers and saves raw matrices and heatmaps.

from pathlib import Path
import gc
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASELINE_HIDDEN_STATES_PATH = Path("data/baseline_hidden_states.pt")
OURS_HIDDEN_STATES_PATH = Path("data/ours_hidden_states.pt")
OUTPUT_DIR = Path("outputs/svcca")
LAYERS = [5, 10, 15, 20, 25, 30, 32]
PCA_COMPONENTS = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16
PROMPT = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n### Instruction: Please answer the following question with true or false, question: do the courts have the power to overrule the laws written by the legislative branch of government?\n\nAnswer format: true/false\n### Response:"
BASELINE_MODEL_SPEC = {"base_model_name": "meta-llama/Meta-Llama-3-8B-Instruct", "adapter_path": Path("adapters/baseline")}
OURS_MODEL_SPEC = {"base_model_name": "meta-llama/Meta-Llama-3-8B-Instruct", "adapter_path": Path("adapters/ours")}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def positive_definite_matrix_sqrt(array):
    w, v = np.linalg.eigh(array)
    return v @ np.diag(np.sqrt(np.maximum(w, 0))) @ np.conj(v).T

def remove_small(sigma_xx, sigma_xy, sigma_yx, sigma_yy, epsilon):
    x_diag = np.abs(np.diagonal(sigma_xx))
    y_diag = np.abs(np.diagonal(sigma_yy))
    x_idxs = x_diag >= epsilon
    y_idxs = y_diag >= epsilon
    return sigma_xx[x_idxs][:, x_idxs], sigma_xy[x_idxs][:, y_idxs], sigma_yx[y_idxs][:, x_idxs], sigma_yy[y_idxs][:, y_idxs], x_idxs, y_idxs

def compute_ccas(sigma_xx, sigma_xy, sigma_yx, sigma_yy, epsilon):
    if sigma_xx.size == 0 or sigma_yy.size == 0:
        return np.array([], dtype=np.float64)
    sigma_xx = sigma_xx + epsilon * np.eye(sigma_xx.shape[0])
    sigma_yy = sigma_yy + epsilon * np.eye(sigma_yy.shape[0])
    inv_xx = np.linalg.pinv(positive_definite_matrix_sqrt(sigma_xx))
    inv_yy = np.linalg.pinv(positive_definite_matrix_sqrt(sigma_yy))
    arr = inv_xx @ sigma_xy @ inv_yy
    _, coef, _ = np.linalg.svd(arr)
    return coef

def get_cca_similarity(acts1, acts2, epsilon=1e-10):
    acts1 = np.asarray(acts1, dtype=np.float64)
    acts2 = np.asarray(acts2, dtype=np.float64)
    if acts1.ndim != 2 or acts2.ndim != 2:
        raise ValueError("CCA inputs must be 2D arrays")
    if acts1.shape[1] != acts2.shape[1]:
        raise ValueError(f"Activation sample counts must match: {acts1.shape} vs {acts2.shape}")
    if acts1.shape[1] < 2:
        raise ValueError("At least two samples are required for CCA")
    m = acts1.shape[1]
    sigma_xx = acts1 @ acts1.T / (m - 1)
    sigma_xy = acts1 @ acts2.T / (m - 1)
    sigma_yx = sigma_xy.T
    sigma_yy = acts2 @ acts2.T / (m - 1)
    sigma_xx, sigma_xy, sigma_yx, sigma_yy, x_idxs, y_idxs = remove_small(sigma_xx, sigma_xy, sigma_yx, sigma_yy, epsilon)
    coef = compute_ccas(sigma_xx, sigma_xy, sigma_yx, sigma_yy, epsilon)
    return {"cca_coef1": coef, "x_idxs": x_idxs, "y_idxs": y_idxs}

def get_svcca_score(acts1, acts2, n_components=20):
    cacts1 = acts1 - np.mean(acts1, axis=1, keepdims=True)
    cacts2 = acts2 - np.mean(acts2, axis=1, keepdims=True)
    _, s1, v1 = np.linalg.svd(cacts1, full_matrices=False)
    _, s2, v2 = np.linalg.svd(cacts2, full_matrices=False)
    k = min(n_components, len(s1), len(s2))
    if k == 0:
        return np.nan
    svacts1 = np.diag(s1[:k]) @ v1[:k]
    svacts2 = np.diag(s2[:k]) @ v2[:k]
    coef = get_cca_similarity(svacts1, svacts2)["cca_coef1"]
    return float(np.mean(coef)) if coef.size else np.nan

def normalize_hidden_states(value):
    if isinstance(value, dict) and "hidden_states" in value:
        value = value["hidden_states"]
    if isinstance(value, torch.Tensor):
        raise ValueError("Expected a sequence of hidden states, not a single tensor")
    return value

def load_hidden_states(path):
    return normalize_hidden_states(torch.load(path, map_location="cpu"))

def extract_hidden_states(spec):
    base_model_name = spec["base_model_name"]
    adapter_path = spec.get("adapter_path")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=DTYPE, device_map=None)
    if adapter_path is not None:
        if not Path(adapter_path).exists():
            raise FileNotFoundError(f"Adapter path not found: {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path)).merge_and_unload()
    model = model.to(DEVICE).eval()
    inputs = tokenizer(PROMPT, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
    hidden_states = tuple(h.detach().cpu() for h in outputs.hidden_states)
    del model, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return hidden_states

def get_hidden_states(path, spec, cache_path):
    if path.is_file():
        return load_hidden_states(path)
    hidden_states = extract_hidden_states(spec)
    torch.save({"hidden_states": hidden_states}, cache_path)
    return hidden_states

def layer_activation(hidden_states, layer):
    hidden_states = normalize_hidden_states(hidden_states)
    if len(hidden_states) == 0:
        raise ValueError("Hidden states are empty")
    first = hidden_states[0]
    if isinstance(first, (tuple, list)):
        value = hidden_states[0][layer]
    else:
        value = hidden_states[layer]
    if not isinstance(value, torch.Tensor):
        value = torch.as_tensor(value)
    if value.ndim == 3:
        value = value.squeeze(0)
    if value.ndim != 2:
        raise ValueError(f"Expected [tokens, hidden_dim] for layer {layer}, got shape {tuple(value.shape)}")
    return value.T.cpu().numpy()

def svcca_matrix(hidden_states_a, hidden_states_b, layers, n_components=20):
    matrix = np.zeros((len(layers), len(layers)), dtype=np.float64)
    for i, layer_a in enumerate(layers):
        for j, layer_b in enumerate(layers):
            acts1 = layer_activation(hidden_states_a, layer_a)
            acts2 = layer_activation(hidden_states_b, layer_b)
            matrix[i, j] = get_svcca_score(acts1, acts2, n_components)
    return matrix

def plot_heatmap(matrix, output_path, x_label="Layer", y_label="Layer", normalize=False):
    values = matrix.copy()
    if normalize:
        denom = np.nanmax(values) - np.nanmin(values)
        values = (values - np.nanmin(values)) / denom if denom else values
    plt.figure(figsize=(8, 6))
    sns.heatmap(values, xticklabels=LAYERS, yticklabels=LAYERS, annot=True, fmt=".5f", cmap="magma", cbar=True)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()

In [ ]:
baseline_hidden_states = get_hidden_states(BASELINE_HIDDEN_STATES_PATH, BASELINE_MODEL_SPEC, OUTPUT_DIR / "baseline_hidden_states.pt")
ours_hidden_states = get_hidden_states(OURS_HIDDEN_STATES_PATH, OURS_MODEL_SPEC, OUTPUT_DIR / "ours_hidden_states.pt")

baseline_matrix = svcca_matrix(baseline_hidden_states, baseline_hidden_states, LAYERS, PCA_COMPONENTS)
cross_matrix = svcca_matrix(baseline_hidden_states, ours_hidden_states, LAYERS, PCA_COMPONENTS)
ours_matrix = svcca_matrix(ours_hidden_states, ours_hidden_states, LAYERS, PCA_COMPONENTS)

np.save(OUTPUT_DIR / "svcca_baseline.npy", baseline_matrix)
np.save(OUTPUT_DIR / "svcca_baseline_vs_ours.npy", cross_matrix)
np.save(OUTPUT_DIR / "svcca_ours.npy", ours_matrix)

plot_heatmap(baseline_matrix, OUTPUT_DIR / "svcca_baseline.png")
plot_heatmap(cross_matrix, OUTPUT_DIR / "svcca_baseline_vs_ours.png", x_label="Ours", y_label="Baseline")
plot_heatmap(ours_matrix, OUTPUT_DIR / "svcca_ours.png")
plot_heatmap(cross_matrix, OUTPUT_DIR / "svcca_baseline_vs_ours_normalized.png", x_label="Ours", y_label="Baseline", normalize=True)